In [0]:
CREATE OR REPLACE TABLE workspace.lakehouse_demo.delta_orders_practice
USING DELTA AS
SELECT *
FROM VALUES
  ('ORD001', 'success', 100.0),
  ('ORD002', 'failed',   200.0),
  ('ORD003', 'success', 150.0)
AS t(order_id, status, order_amount);

num_affected_rows,num_inserted_rows


In [0]:
SELECT *
FROM workspace.lakehouse_demo.delta_orders_practice
ORDER BY order_id;

order_id,status,order_amount
ORD001,success,100.0
ORD002,failed,200.0
ORD003,success,150.0


In [0]:
DESCRIBE HISTORY workspace.lakehouse_demo.delta_orders_practice;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
0,2026-10-03T23:08:28.000Z,74861291488248,abdounekhadija358@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""io.unitycatalog.tableId"":""09df5f02-3efd-4137-ac18-1d0da1e1f084""}, statsOnLoad -> true)",null,List(693970517020196),8b8f4ddf-9cac-404b-8db2-99f55f436d90,1003-223732-xx2glm0y-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 3, numOutputBytes -> 1361)",null,Databricks-Runtime/19.8.x-aarch64-photon-scala2.13


In [0]:
UPDATE workspace.lakehouse_demo.delta_orders_practice
SET status = 'success'
WHERE order_id = 'ORD002';

num_affected_rows
1


In [0]:
SELECT *
FROM workspace.lakehouse_demo.delta_orders_practice
ORDER BY order_id;

order_id,status,order_amount
ORD001,success,100.0
ORD002,success,200.0
ORD003,success,150.0


In [0]:
SELECT *
FROM workspace.lakehouse_demo.delta_orders_practice VERSION AS OF 0
ORDER BY order_id;

order_id,status,order_amount
ORD001,success,100.0
ORD002,failed,200.0
ORD003,success,150.0


In [0]:
DESCRIBE HISTORY workspace.lakehouse_demo.delta_orders_practice;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
2,2026-10-03T23:10:47.000Z,74861291488248,abdounekhadija358@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(693970517020196),77d22837-85e2-4cda-82d8-53d3152e6eef,1003-223732-xx2glm0y-v2n,1,SnapshotIsolation,false,"Map(numRemovedFiles -> 2, numRemovedBytes -> 2675, p25FileSize -> 1388, numDeletionVectorsRemoved -> 1, minFileSize -> 1388, numAddedFiles -> 1, maxFileSize -> 1388, p75FileSize -> 1388, p50FileSize -> 1388, numAddedBytes -> 1388)",null,Databricks-Runtime/19.8.x-aarch64-photon-scala2.13
1,2026-10-03T23:10:44.000Z,74861291488248,abdounekhadija358@gmail.com,UPDATE,"Map(predicate -> [""(order_id#11040 = ORD002)""])",null,List(693970517020196),77d22837-85e2-4cda-82d8-53d3152e6eef,1003-223732-xx2glm0y-v2n,0,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 5242, numDeletionVectorsUpdated -> 0, scanTimeMs -> 2544, numAddedFiles -> 1, numUpdatedRows -> 1, numAddedBytes -> 1314, rewriteTimeMs -> 2666)",null,Databricks-Runtime/19.8.x-aarch64-photon-scala2.13
0,2026-10-03T23:08:28.000Z,74861291488248,abdounekhadija358@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""io.unitycatalog.tableId"":""09df5f02-3efd-4137-ac18-1d0da1e1f084""}, statsOnLoad -> true)",null,List(693970517020196),8b8f4ddf-9cac-404b-8db2-99f55f436d90,1003-223732-xx2glm0y-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 3, numOutputBytes -> 1361)",null,Databricks-Runtime/19.8.x-aarch64-photon-scala2.13


In [0]:
CREATE OR REPLACE TEMP VIEW incoming_orders AS
SELECT *
FROM VALUES
  ('ORD003', 'refunded', 150.0),
  ('ORD004', 'success',  300.0),
  ('ORD005', 'failed',    80.0)
AS t(order_id, status, order_amount);

In [0]:
MERGE INTO workspace.lakehouse_demo.delta_orders_practice AS target
USING incoming_orders AS source
ON target.order_id = source.order_id

WHEN MATCHED AND (
  NOT (target.status <=> source.status)
  OR NOT (target.order_amount <=> source.order_amount)
)
THEN UPDATE SET
  target.status = source.status,
  target.order_amount = source.order_amount

WHEN NOT MATCHED THEN
INSERT (order_id, status, order_amount)
VALUES (source.order_id, source.status, source.order_amount);

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
3,1,0,2


In [0]:
SELECT *
FROM workspace.lakehouse_demo.delta_orders_practice
ORDER BY order_id;

order_id,status,order_amount
ORD001,success,100.0
ORD002,success,200.0
ORD003,refunded,150.0
ORD004,success,300.0
ORD005,failed,80.0
